# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: K-Means clustering**, per the framing table — "What kinds of items exist?" → Clustering → no target → silhouette + human sense-check. This is the right fit for two reasons:

* Your lane was never "predict decline" (that would need an observed future outcome you don't have — you flagged this exact gap at the end of ML-07). It was always "what types of pages exist," which is unsupervised by definition.
* Logistic Regression / Random Forest / Gradient Boosting all need a label. The only label-shaped thing available (`trend_direction`) is excluded as a feature by the flyrank-data label trap, and using it as a target would just teach a model to reproduce a threshold rule on `trend_pct` — not a real-world outcome. That's the "defined, not observed" trap the framing skill warns about. So classification isn't honest here yet; clustering is.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Features: structural + performance signals only — no trend_direction/trend_pct (label trap),
# no content_id/client_id (context, not features)
feature_cols = [
    "search_volume", "competition", "cpc", "word_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "content_age_days", "days_since_last_update"
]

X = df[feature_cols].copy()
print("Missing values per feature:\n", X.isna().sum()[X.isna().sum() > 0])
X = X.fillna(X.median(numeric_only=True))  # median, not 0 — these are skewed counts, not "has_" flags

Missing values per feature:
 search_volume    2468
competition      2468
cpc              2468
word_count       7699
scroll_rate       125
dtype: int64


In [11]:
print(df.groupby("content_type")[["search_volume", "word_count", "scroll_rate"]].apply(lambda g: g.isna().mean()))

                    search_volume  word_count  scroll_rate
content_type                                              
comparison article       0.000000    0.000000     0.002869
feedly article           1.000000    0.000000     0.000477
keyword article          0.013673    0.282979     0.004484


In [10]:
# Instead of blind median fillna, add has_-flags for structurally-missing fields,
# then fill with a neutral value that won't be mistaken for a real measurement
for col in ["search_volume", "competition", "cpc", "word_count"]:
    X[f"has_{col}"] = df[col].notna().astype(int)
    X[col] = X[col].fillna(0)  # 0 here is fine ONLY because has_-flag preserves the distinction

# scroll_rate: small, likely sporadic — median fill without a flag is defensible,
# but confirm it isn't content_type-patterned first (see check above)
X["scroll_rate"] = X["scroll_rate"].fillna(X["scroll_rate"].median())

feature_cols = feature_cols + ["has_search_volume", "has_competition", "has_cpc", "has_word_count"]

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, not random.** Per flyrank-data: `client_id` should be used for grouped splits, never as a feature. The reason it matters here specifically — clients differ wildly in scale (a big client's "typical" page might dwarf a small client's "champion" page), so a random split would let the same client's pages leak into both a fit set and a held-out stability check, making the clusters look artificially more stable than they'd be on a genuinely new client.

In [19]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
fit_idx, holdout_idx = next(gss.split(X, groups=df["client_id"]))

X_fit, X_holdout = X.iloc[fit_idx], X.iloc[holdout_idx]
print(f"Fit set: {len(X_fit)} rows, {df.iloc[fit_idx]['client_id'].nunique()} clients")
print(f"Holdout set: {len(X_holdout)} rows, {df.iloc[holdout_idx]['client_id'].nunique()} clients")
# confirm no client appears in both
overlap = set(df.iloc[fit_idx]["client_id"]) & set(df.iloc[holdout_idx]["client_id"])
print("Client overlap between fit/holdout (should be empty):", overlap)

Fit set: 23837 rows, 25 clients
Holdout set: 6163 rows, 7 clients
Client overlap between fit/holdout (should be empty): set()


In [16]:
print(df.groupby("client_id").size().sort_values(ascending=False))

client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031
client_d029fa3a95     952
client_349c41201b     763
client_e629fa6598     720
client_e29c9c180c     708
client_2c624232cd     649
client_8722616204     578
client_4ec9599fc2     556
client_bbb965ab0c     505
client_25fc0e7096     476
client_624b60c58c     337
client_9f14025af0     325
client_b4944c6ff0     254
client_9400f1b21c     158
client_98a3ab7c34     118
client_434c9b5ae5      87
client_bdd2d3af3a      44
client_d59eced1de      43
client_02d20bbd7e      38
client_0b918943df      35
client_4fc82b26ae      32
client_8b940be7fb      28
client_1a6562590e       3
dtype: int64


In [20]:
# Does the split ratio hold up across different random seeds, or was 42 lucky?
for seed in [0, 1, 2, 42, 100]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    f_idx, h_idx = next(gss.split(X, groups=df["client_id"]))
    print(f"seed={seed}: fit={len(f_idx)} ({len(f_idx)/len(df):.1%}), holdout={len(h_idx)} ({len(h_idx)/len(df):.1%})")

seed=0: fit=19821 (66.1%), holdout=10179 (33.9%)
seed=1: fit=27838 (92.8%), holdout=2162 (7.2%)
seed=2: fit=22402 (74.7%), holdout=7598 (25.3%)
seed=42: fit=23837 (79.5%), holdout=6163 (20.5%)
seed=100: fit=26287 (87.6%), holdout=3713 (12.4%)


**Split: grouped by** `client_id`, **not random**, using `GroupShuffleSplit`, to avoid the same client appearing in both fit and holdout (per flyrank-data: `client_id` is for grouping/splitting, never a feature). Confirmed zero client overlap.

**Limitation, quantified rather than assumed:** client sizes are highly skewed (largest client = 7,008 rows / 23% of the dataset; smallest = 3 rows). A seed-sensitivity check across 5 seeds shows the resulting fit/holdout ratio swings from 66/34 to 93/7, despite requesting an 80/20 split every time — because whichever large client lands in holdout dominates the ratio. Seed 42 (79.5/20.5) is used for the rest of this notebook, but this is a lucky draw, not a guaranteed property of the split. A more rigorous fix (stratifying the group split by client-size bucket) is left as future work rather than implemented this week.

This also explains the ML-07 finding that one client supplied 11 of 14 top-priority baseline pages: that client is a small slice of the dataset (1,043 rows, ~3.5%), so its dominance in that ranking reflects a client-specific staleness pattern, not a dataset-wide one.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Here's the honest reframe for this section: precision@K doesn't transfer to clustering, so the comparison isn't "does K-Means beat the rule at the same metric" — it's "**does clustering add resolution the hand-written rule doesn't have.**" Your ML-07 baseline sorted every page into 4 buckets by hand. K-Means finds structure without being told the buckets. The fair comparison is whether clustering reveals meaningful sub-structure inside those buckets — especially inside `low_priority`, which swallowed 21,085 of your 30,000 pages with zero further distinction.

In [22]:
feature_cols = [
    "search_volume", "competition", "cpc", "word_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "engaged_sessions_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "content_age_days", "days_since_last_update"
]

X = df[feature_cols].copy()

# has_-flags for structurally-missing (content_type-driven) fields — confirmed pattern
for col in ["search_volume", "competition", "cpc", "word_count"]:
    X[f"has_{col}"] = df[col].notna().astype(int)
    X[col] = X[col].fillna(0)

# scroll_rate: sporadic, not structural — median fill, no flag needed
X["scroll_rate"] = X["scroll_rate"].fillna(X["scroll_rate"].median())

feature_cols = feature_cols + ["has_search_volume", "has_competition", "has_cpc", "has_word_count"]

# --- hard checkpoint: prove there's no NaN left before touching KMeans ---
remaining_nan = X.isna().sum()
assert remaining_nan.sum() == 0, f"Still have NaN in: {remaining_nan[remaining_nan > 0]}"
print("Confirmed: zero NaN remaining across all", len(feature_cols), "features.")

# --- split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
fit_idx, holdout_idx = next(gss.split(X, groups=df["client_id"]))
X_fit, X_holdout = X.iloc[fit_idx], X.iloc[holdout_idx]

# --- scale ---
scaler = StandardScaler()
X_fit_scaled = scaler.fit_transform(X_fit)
X_holdout_scaled = scaler.transform(X_holdout)

print("X_fit_scaled NaN count:", pd.DataFrame(X_fit_scaled).isna().sum().sum())

Confirmed: zero NaN remaining across all 19 features.
X_fit_scaled NaN count: 0


In [23]:
scaler = StandardScaler()
X_fit_scaled = scaler.fit_transform(X_fit)
X_holdout_scaled = scaler.transform(X_holdout)  # transform only, never re-fit on holdout

# pick k with silhouette
for k in range(3, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_fit_scaled)
    sil = silhouette_score(X_fit_scaled, labels)
    print(f"k={k}: silhouette={sil:.3f}")

k=3: silhouette=0.286
k=4: silhouette=0.297
k=5: silhouette=0.296
k=6: silhouette=0.261
k=7: silhouette=0.268


In [25]:
# recreate reason_code from ML-07 baseline logic, so this notebook is self-contained
visibility_bar = df["impressions_90d"].quantile(0.5)

def reason_code(row):
    if row["trend_direction"] == "down" and row["impressions_90d"] >= visibility_bar:
        return "stale_declining_visible" if row["days_since_last_update"] >= 180 else "declining_visible_fresh"
    if row["impressions_90d"] >= visibility_bar and row["days_since_last_update"] >= 180:
        return "stale_visible_stable"
    return "low_priority"

df["reason_code"] = df.apply(reason_code, axis=1)
print(df["reason_code"].value_counts())

reason_code
low_priority               21085
declining_visible_fresh     8900
stale_declining_visible       14
stale_visible_stable           1
Name: count, dtype: int64


In [26]:
FINAL_K = 4  # replace with whatever wins above
km = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
df.loc[X_fit.index, "cluster"] = km.fit_predict(X_fit_scaled)
df.loc[X_holdout.index, "cluster"] = km.predict(X_holdout_scaled)

holdout_sil = silhouette_score(X_holdout_scaled, df.loc[X_holdout.index, "cluster"])
print(f"Holdout silhouette (stability check): {holdout_sil:.3f}")

# --- The actual "vs baseline" comparison table ---
comparison = pd.crosstab(df["reason_code"], df["cluster"])
print("\nBaseline reason_code vs K-Means cluster (does clustering split the buckets further?):")
print(comparison)

print(f"\n| Metric | Baseline (4 hand-written buckets) | K-Means (k={FINAL_K}) |")
print(f"|---|---|---|")
print(f"| Groups found | 4 | {FINAL_K} |")
print(f"| Resolution inside 'low_priority' (21,085 rows) | 1 undifferentiated bucket | {comparison.loc['low_priority'].gt(0).sum()} distinct clusters |")
print(f"| Metric | none defined (rule-based) | silhouette = {holdout_sil:.3f} (holdout) |")

Holdout silhouette (stability check): 0.278

Baseline reason_code vs K-Means cluster (does clustering split the buckets further?):
cluster                    0.0   1.0  2.0   3.0
reason_code                                    
declining_visible_fresh   6612  2035  145   108
low_priority             13009  5480  237  2359
stale_declining_visible     12     0    0     2
stale_visible_stable         1     0    0     0

| Metric | Baseline (4 hand-written buckets) | K-Means (k=4) |
|---|---|---|
| Groups found | 4 | 4 |
| Resolution inside 'low_priority' (21,085 rows) | 1 undifferentiated bucket | 4 distinct clusters |
| Metric | none defined (rule-based) | silhouette = 0.278 (holdout) |


In [28]:
cluster_profile = df.groupby("cluster")[[
    "sessions_90d", "engagement_rate", "impressions_90d",
    "content_age_days", "days_since_last_update", "avg_position", "ai_traffic_pct"
]].median()
print(cluster_profile)

print("\nCluster sizes:")
print(df["cluster"].value_counts().sort_index())

         sessions_90d  engagement_rate  impressions_90d  content_age_days  \
cluster                                                                     
0.0               7.0             0.00            871.5             155.0   
1.0               9.0             0.00            872.0             438.0   
2.0             503.0             2.63          80667.0             230.0   
3.0               3.0             0.00              6.0             296.0   

         days_since_last_update  avg_position  ai_traffic_pct  
cluster                                                        
0.0                        20.0         11.70             0.0  
1.0                        22.0         11.80             0.0  
2.0                       104.0          5.85             0.0  
3.0                        20.0          4.80             0.0  

Cluster sizes:
cluster
0.0    19634
1.0     7515
2.0      382
3.0     2469
Name: count, dtype: int64


K-Means (k=4, chosen by best silhouette at 0.297, holding up at 0.278 on the grouped holdout) found some real structure beyond the baseline's four hand-written buckets, but the added resolution is uneven. The baseline's `low_priority` bucket (21,085 rows, previously undifferentiated) split across all four clusters, with 2,359 of those rows landing in a distinct low-performing cluster and 237 in a small high-performing one — genuine sub-structure the rule-based baseline couldn't see. However, one cluster (cluster 0) still holds 61% of the full dataset on its own, so this is better described as partial added resolution than a clean four-way split. Because clustering has no shared metric with the baseline's precision@K, this comparison is reported as structural resolution gained, not a performance win — a directional, decision-support result rather than a validated improvement.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [29]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.inspection import permutation_importance

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_fit, df.loc[X_fit.index, "cluster"])
print(export_text(tree, feature_names=list(X_fit.columns)))

perm = permutation_importance(tree, X_holdout, df.loc[X_holdout.index, "cluster"], n_repeats=10, random_state=42)
importance = pd.Series(perm.importances_mean, index=X_holdout.columns).sort_values(ascending=False)
print("\nTop features driving cluster assignment:\n", importance.head(6))

|--- has_word_count <= 0.50
|   |--- has_competition <= 0.50
|   |   |--- sessions_90d <= 435.00
|   |   |   |--- class: 3.0
|   |   |--- sessions_90d >  435.00
|   |   |   |--- class: 2.0
|   |--- has_competition >  0.50
|   |   |--- sessions_90d <= 299.50
|   |   |   |--- class: 1.0
|   |   |--- sessions_90d >  299.50
|   |   |   |--- class: 2.0
|--- has_word_count >  0.50
|   |--- has_search_volume <= 0.50
|   |   |--- class: 3.0
|   |--- has_search_volume >  0.50
|   |   |--- clicks_90d <= 226.00
|   |   |   |--- class: 0.0
|   |   |--- clicks_90d >  226.00
|   |   |   |--- class: 2.0


Top features driving cluster assignment:
 has_word_count       0.271329
has_search_volume    0.028265
has_competition      0.018627
clicks_90d           0.009622
sessions_90d         0.002093
search_volume        0.000000
dtype: float64


The decision tree reading the clusters back out shows the split is driven almost entirely by has_word_count (importance 0.271, by far the top feature), with has_search_volume and has_competition a distant second and third — actual performance signals like sessions_90d and clicks_90d only matter within those missingness splits, not as the primary driver. Since has_word_count/has_search_volume were confirmed earlier to be near-perfect proxies for content_type (feedly article vs keyword article vs comparison article), this means the clustering is substantially rediscovering known content categories, not finding a new behavioral archetype. That's an honest limitation to name directly: clusters 2 and 3 (the two clearest, most actionable groups — champions and weak/no-demand) are real and useful, but the broader four-cluster split should be described as "content-type-driven, with performance-based sub-splits inside each type" rather than four independently discovered archetypes. This is a legitimate, decision-support finding on its own — it tells the content team that page type, not just page behavior, structurally shapes what data is even available to judge a page by — but it is not the purely behavioral clustering the lane originally set out to find, and the notebook should say so plainly rather than claim more than the evidence supports.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.